## Table of Contents

- [Introduction to LLMs](#introduction-to-llms)
  - [Tokenizer](#tokenizer)
  - [Next-Token Predictors](#next-token-predictors)
- [Transformer Intuition](#transformer-intuition)
- [Tokenization](#tokenization)
- [Embeddings](#embeddings)
- [Positional Encoding](#positional-encoding)
- [Multi-Head Attention](#multi-head-attention)
- [Feed Forward Network](#feed-forward-network)
- [Encoder & Decoder Blocks](#encoder--decoder-blocks)
- [Full Transformer Assembly](#full-transformer-assembly)
- [Training Loop](#training-loop)
- [Loss Function & Optimization](#loss-function--optimization)
- [Fine-Tuning (High Level)](#fine-tuning-high-level)


## Introduction to LLMs

A large language model (LLM) looks like it “understands” text, but the core training objective is simple:
**predict the next token**.

If we represent a piece of text as a sequence of tokens:

$$t_1, t_2, \dots, t_n$$

then an autoregressive (causal) language model learns the conditional distribution:

$$p(t_i \mid t_1, \dots, t_{i-1})$$

During training, we show the model many examples and push it to assign high probability to the correct next token.
During generation, we repeatedly apply the model and append predicted tokens one-by-one.

### Tokenizer

Neural networks can’t directly process raw text. We first convert text into **token IDs** (integers).
That conversion is handled by the tokenizer. Depending on design, tokens can be:

- words
- subwords
- characters

Example: “Hold my math!”

- Word-level: `["Hold", "my", "math", "!"]`
- Subword-level: `["Hold", "my", "ma", "th", "!"]`
- Character-level: `["H","o","l","d"," ","m","y"," ","m","a","t","h","!"]`

In this notebook we implement a *character-level* tokenizer (fully from scratch) because it’s the most transparent for learning.

### Next-Token Predictors

Given a prefix of tokens, a language model outputs a probability distribution over the vocabulary for the next token.

![Next-token predictor](images/next_token_predictor.png)

Generation is simply repeating the same step in a loop: predict → sample → append.

![Autoregressive window](images/autoregressive_window.png)

Practical models can only process a fixed maximum number of tokens at once (the **context window**),
so generation always conditions on a bounded window of previous tokens.


### A quick note on chat formatting

Chat APIs often accept a list of messages like `{role, content}`. Internally, many systems serialize these into
a single text prompt with special boundary markers. The exact markers vary by model family, but the idea is the same:
teach the model to separate **system** behavior, **user** queries, and **assistant** responses.


In [ ]:
messages = [
    {"role": "system", "content": "You are a creative storyteller."},
    {"role": "user", "content": "Write a creative story"},
]

serialized = (
    "<|im_start|>system\n"
    "You are a creative storyteller.\n"
    "<|im_end|>\n"
    "<|im_start|>user\n"
    "Write a creative story\n"
    "<|im_end|>\n"
    "<|im_start|>assistant\n"
)

messages, serialized


## Transformer Intuition

Early sequence models like RNNs process tokens one at a time, passing a hidden state forward.
That helps incorporate “past context”, but creates two big problems:

1. **Long-range dependencies are fragile**: older information can fade as sequences get longer.
2. **Training is hard to parallelize**: each step depends on the previous step.

![RNN overview](images/rnn.png)

LSTMs (with gates) improved memory retention, but the sequential bottleneck still remains.

![LSTM overview](images/lstm.png)

Transformers replace recurrence with **attention**, which connects every token to every other token directly.
Instead of “carrying memory forward”, each token computes how much it should “pay attention” to other tokens.

### The attention computation

Given token representations $X \in \mathbb{R}^{T \times d_{model}}$, we project each token into:

- Query $Q$ (what I’m looking for)
- Key $K$ (what I offer)
- Value $V$ (what I will contribute if attended to)

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$

Then scaled dot-product attention is:

$$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

**💡 Insight**

Attention is content-based routing: tokens choose what information to mix in by comparing queries to keys,
and then averaging values using those similarity scores.


In [1]:
import math
import random
from dataclasses import dataclass
from typing import List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed: int = 1337) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(1337)
print("device:", device)


device: cpu


## Tokenization

We’ll implement a character-level tokenizer:

- Build a vocabulary from the training corpus.
- Encode: text → list of integers.
- Decode: list of integers → text.

This keeps the mechanics crystal clear and requires no external tokenization libraries.


In [2]:
class CharTokenizer:
    def __init__(self, text: str, add_special_tokens: bool = True):
        self.special_tokens = ["<pad>", "<unk>", "<bos>", "<eos>"] if add_special_tokens else []

        chars = sorted(list(set(text)))
        self.itos = self.special_tokens + chars
        self.stoi = {ch: i for i, ch in enumerate(self.itos)}

        self.pad_id = self.stoi.get("<pad>")
        self.unk_id = self.stoi.get("<unk>")
        self.bos_id = self.stoi.get("<bos>")
        self.eos_id = self.stoi.get("<eos>")

    @property
    def vocab_size(self) -> int:
        return len(self.itos)

    def encode(self, s: str, add_bos: bool = False, add_eos: bool = False) -> List[int]:
        ids: List[int] = []
        if add_bos and self.bos_id is not None:
            ids.append(self.bos_id)
        for ch in s:
            ids.append(self.stoi.get(ch, self.unk_id))
        if add_eos and self.eos_id is not None:
            ids.append(self.eos_id)
        return ids

    def decode(self, ids: List[int], strip_special: bool = True) -> str:
        out: List[str] = []
        for i in ids:
            if i < 0 or i >= len(self.itos):
                continue
            tok = self.itos[i]
            if strip_special and tok in self.special_tokens:
                continue
            out.append(tok)
        return "".join(out)

sample = "Hold my math!"
tok = CharTokenizer(sample)
encoded = tok.encode(sample, add_bos=True, add_eos=True)
print("vocab_size:", tok.vocab_size)
print("encoded:", encoded)
print("decoded:", tok.decode(encoded))


vocab_size: 15
encoded: [2, 6, 12, 10, 8, 4, 11, 14, 4, 11, 7, 13, 9, 5, 3]
decoded: Hold my math!


## Embeddings

Token IDs are discrete integers; Transformers operate on vectors.
So we learn a token embedding table $E \in \mathbb{R}^{V \times d_{model}}$, where:

- $V$ is vocabulary size
- $d_{model}$ is the embedding dimension

Each token id indexes a row in $E$ to produce a vector representation.


In [3]:
V = tok.vocab_size
D = 16
embedding = nn.Embedding(V, D)

ids = torch.tensor(tok.encode(sample), dtype=torch.long)
vecs = embedding(ids)

print("ids shape:", tuple(ids.shape))
print("vecs shape:", tuple(vecs.shape))


ids shape: (13,)
vecs shape: (13, 16)


## Positional Encoding

Without recurrence or convolution, a Transformer has no built-in notion of order.
We inject position information via **positional encodings**.

The sinusoidal encoding from the original Transformer paper is:

$$PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

We add positional encodings to token embeddings:

$$X = \mathrm{Embed}(tokens) + PE$$

so the model can reason about both meaning and order.


In [4]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        T = x.size(1)
        return x + self.pe[:, :T, :]

pos_enc = SinusoidalPositionalEncoding(16, max_len=64)
print(pos_enc(torch.zeros(2, 10, 16)).shape)


torch.Size([2, 10, 16])


## Multi-Head Attention

Self-attention works by comparing every token to every other token.
For a single head:

$$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

**Why the scaling term?**
The dot product $QK^\top$ grows with $d_k$. Scaling by $\sqrt{d_k}$ keeps softmax in a healthy range
and prevents tiny gradients.

**Why multiple heads?**
Multi-head attention runs several attention computations in parallel, each on a smaller subspace.
Different heads can specialize in different patterns (syntax, long-range dependencies, matching brackets, etc.).

**⚠️ Common Mistakes**

- Forgetting the causal mask for language modeling (it leaks future tokens).
- Applying softmax over the wrong axis (must normalize over key positions).
- Mixing up the `(B, T, h, d_head)` reshapes.


In [5]:
def make_causal_mask(T: int, device=None) -> torch.Tensor:
    m = torch.tril(torch.ones(T, T, dtype=torch.bool, device=device))
    return m.view(1, 1, T, T)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0, is_causal: bool = False):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.is_causal = is_causal

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        kv: torch.Tensor | None = None,
        attn_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        B, Tq, _ = x.shape
        if kv is None:
            kv = x
        _, Tk, _ = kv.shape

        q = self.q_proj(x)
        k = self.k_proj(kv)
        v = self.v_proj(kv)

        q = q.view(B, Tq, self.n_heads, self.d_head).transpose(1, 2)  # (B, h, Tq, d_head)
        k = k.view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)  # (B, h, Tk, d_head)
        v = v.view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)  # (B, h, Tk, d_head)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)  # (B, h, Tq, Tk)

        if self.is_causal:
            causal = make_causal_mask(Tq, device=scores.device)
            scores = scores.masked_fill(~causal, float("-inf"))

        if attn_mask is not None:
            scores = scores.masked_fill(~attn_mask, float("-inf"))

        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        out = attn @ v  # (B, h, Tq, d_head)
        out = out.transpose(1, 2).contiguous().view(B, Tq, self.d_model)
        return self.out_proj(out)

mha = MultiHeadAttention(d_model=32, n_heads=4, dropout=0.1, is_causal=True)
print(mha(torch.randn(2, 8, 32)).shape)


torch.Size([2, 8, 32])


**Explanation**

- We project inputs into Q/K/V of shape `(B, T, d_model)`.
- We split `d_model` into `n_heads` smaller heads of size `d_head`.
- We compute attention scores for each head: `(B, h, Tq, Tk)`.
- Softmax runs over `Tk` so weights sum to 1 across keys for each query token.
- A causal mask ensures token *i* can’t attend to tokens *j > i*.


## Feed Forward Network

Attention mixes information across tokens. The feed-forward network (FFN) then transforms each token’s representation independently.

$$\mathrm{FFN}(x) = W_2\,\sigma(W_1x + b_1) + b_2$$

Typically the hidden dimension is ~4× larger than $d_{model}$. We’ll use GELU activation.


In [6]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

print(FeedForward(32, 128, 0.1)(torch.randn(2, 8, 32)).shape)


torch.Size([2, 8, 32])


## Encoder & Decoder Blocks

A full “classic” Transformer has:

- **Encoder blocks** (self-attention + FFN)
- **Decoder blocks** (masked self-attention + cross-attention + FFN)

Many LLMs (GPT-style) are **decoder-only** (masked self-attention + FFN) because next-token prediction needs only causal context.

We’ll implement both:

- an encoder block (for completeness and conceptual alignment)
- a decoder block (used by our language model)

We’ll use **Pre-LN** (LayerNorm before each sub-layer), which is common in modern training for stability.


In [7]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout=dropout, is_causal=False)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout=dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class DecoderBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout=dropout, is_causal=True)

        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout=dropout, is_causal=False)

        self.ln3 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout=dropout)

    def forward(self, x: torch.Tensor, enc_out: torch.Tensor | None = None) -> torch.Tensor:
        x = x + self.self_attn(self.ln1(x))
        if enc_out is not None:
            x = x + self.cross_attn(self.ln2(x), kv=enc_out)
        x = x + self.ffn(self.ln3(x))
        return x


## Full Transformer Assembly

For language modeling, we build a **decoder-only Transformer**:

1. Token embedding + positional encoding
2. Stack of masked decoder blocks
3. Output projection to vocabulary size

The model produces logits:

$$\text{logits} \in \mathbb{R}^{B \times T \times V}$$

and training uses cross-entropy against the shifted targets.


In [8]:
@dataclass
class TransformerLMConfig:
    vocab_size: int
    d_model: int = 192
    n_heads: int = 6
    n_layers: int = 4
    d_ff: int = 768
    dropout: float = 0.1
    max_len: int = 128

class TransformerDecoderOnlyLM(nn.Module):
    def __init__(self, cfg: TransformerLMConfig):
        super().__init__()
        self.cfg = cfg
        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_enc = SinusoidalPositionalEncoding(cfg.d_model, max_len=cfg.max_len)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList(
            [DecoderBlock(cfg.d_model, cfg.n_heads, cfg.d_ff, cfg.dropout) for _ in range(cfg.n_layers)]
        )
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        self.lm_head.weight = self.token_emb.weight

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        assert T <= self.cfg.max_len
        x = self.token_emb(idx)
        x = self.pos_enc(x)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x, enc_out=None)
        x = self.ln_f(x)
        return self.lm_head(x)

cfg = TransformerLMConfig(vocab_size=tok.vocab_size, max_len=128)
model = TransformerDecoderOnlyLM(cfg).to(device)
test_ids = torch.tensor([tok.encode("Hold my m")[:20]], dtype=torch.long, device=device)
print("logits:", model(test_ids).shape)


logits: torch.Size([1, 9, 15])


## Training Loop

### Data preparation

For causal LM training, we create examples from a long token stream using a fixed block size `T`:

- input `x`: `[t_1, ..., t_T]`
- target `y`: `[t_2, ..., t_{T+1}]`

So the model learns to predict the next token at every position in parallel.


In [11]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, token_ids: List[int], block_size: int):
        super().__init__()
        self.data = torch.tensor(token_ids, dtype=torch.long)
        self.block_size = block_size

    def __len__(self) -> int:
        return max(0, len(self.data) - self.block_size - 1)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + 1 + self.block_size]
        return x, y

corpus = (
    "We were born to make history.\n"
    "Lights will guide you home, and ignite your bones.\n"
    "Nobody said it was easy.\n"
    "I used to rule the world.\n"
    "Look at the stars, look how they shine for you.\n"
) * 100

tokenizer = CharTokenizer(corpus)
all_ids = tokenizer.encode(corpus)

block_size = 128
split = int(0.9 * len(all_ids))
train_ids, val_ids = all_ids[:split], all_ids[split:]

train_ds = TextDataset(train_ids, block_size)
val_ds = TextDataset(val_ids, block_size)

batch_size = 64
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=True)

cfg = TransformerLMConfig(vocab_size=tokenizer.vocab_size, max_len=block_size)
model = TransformerDecoderOnlyLM(cfg).to(device)

sum(p.numel() for p in model.parameters())


2374080

## Loss Function & Optimization

Next-token prediction uses **cross-entropy loss**.
If logits for a token are $z \in \mathbb{R}^{V}$ and the true next token is $y$:

$$\mathcal{L} = -\log(\mathrm{softmax}(z)_y)$$

In practice we compute cross-entropy over the whole batch and all time steps.
For optimization, **AdamW** is a widely used baseline for Transformers.


In [12]:
def evaluate_loss(model: nn.Module, loader, max_batches: int = 50) -> float:
    model.eval()
    losses = []
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if i >= max_batches:
                break
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
            losses.append(loss.item())
    model.train()
    return float(sum(losses) / max(1, len(losses)))

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

def train(model: nn.Module, steps: int = 200, log_every: int = 50) -> None:
    model.train()
    it = iter(train_loader)

    for step in range(1, steps + 1):
        try:
            x, y = next(it)
        except StopIteration:
            it = iter(train_loader)
            x, y = next(it)

        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        if step == 1 or step % log_every == 0:
            val_loss = evaluate_loss(model, val_loader, max_batches=30)
            print(f"step {step:4d} | train_loss {loss.item():.4f} | val_loss {val_loss:.4f}")

train(model)


step    1 | train_loss 137.8873 | val_loss 126.9168
step   50 | train_loss 2.7809 | val_loss 2.0329
step  100 | train_loss 1.7464 | val_loss 1.5955


KeyboardInterrupt: 

**💡 Insight**

With a causal mask, *every* position becomes a supervised learning target. A single batch teaches many “next token” problems in parallel.

**⚠️ Common Mistakes**

- Not shifting targets (`y`) by one position.
- Forgetting `model.eval()` during validation (dropout changes the loss).
- Training without a causal mask (future-token leakage).


In [ ]:
@torch.no_grad()
def generate(
    model: nn.Module,
    tokenizer: CharTokenizer,
    prompt: str,
    max_new_tokens: int = 200,
    temperature: float = 1.0,
) -> str:
    model.eval()
    idx = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.cfg.max_len :]
        logits = model(idx_cond)[:, -1, :] / max(1e-8, temperature)
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)

    return tokenizer.decode(idx[0].tolist())

print(generate(model, tokenizer, prompt="Look at the ", max_new_tokens=200, temperature=0.9))


## Fine-Tuning (High Level)

Pretraining learns general language patterns from large corpora. Fine-tuning adapts a pretrained model to a specific goal.

### Instruction tuning

Fine-tuning on instruction/response data teaches the model to follow user requests instead of only continuing text.
Data is often formatted with role markers (system/user/assistant) so the model learns what kind of continuation is expected.

### Parameter-efficient fine-tuning

Instead of updating all parameters, methods like LoRA introduce small trainable adapters (low-rank matrices),
reducing memory requirements and helping preserve general pretrained knowledge.
